<a href="https://colab.research.google.com/github/hiiamlars/analytical_flexibility_llm_reviews/blob/main/data_generation/tvd_mi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Setup

In [3]:
# @title Dependencies

# Installations of third-party libraries
!pip install openai -q
!pip install anthropic -q
!pip install together -q
!pip install tqdm -q

# First-party imports
import os
import json
import csv
import time
import random
from typing import List, Dict, Any, Optional, Tuple
import numpy as np
import google.colab
from google.colab import userdata
from google.colab import drive

# Third-party imports
import pandas as pd
from openai import OpenAI
from anthropic import Anthropic
from together import Together
from tqdm.notebook import tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.4/469.4 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.5/331.5 kB 3.8 MB/s eta 0:00:00


In [4]:
# @title Path definitions

# Mount GDrive
drive.mount('/content/drive')

# Set working directory
WORKING_DIR = "/content/drive/MyDrive/Analytical_flexibility_llm_reviews/"

# Create working directory (if not already existing)
os.makedirs(WORKING_DIR, exist_ok=True)
print(f"Ensured working directory exists at: {WORKING_DIR}.")

# Define and create PAPER_DIR for paper content
PAPER_DIR = os.path.join(WORKING_DIR, "iclr/final")
os.makedirs(PAPER_DIR, exist_ok=True)
print(f"Ensured raw directory exists at: {PAPER_DIR}.")

# Define and create LLM_DIR for llm based reviews
LLM_DIR = os.path.join(WORKING_DIR, "llm_reviews")
os.makedirs(LLM_DIR, exist_ok=True)
print(f"Ensured llm review directory exists at: {LLM_DIR}.")

# Define and create RES_DIR for the result file
RES_DIR = os.path.join(WORKING_DIR, "tvd_mi")
os.makedirs(RES_DIR, exist_ok=True)
print(f"Ensured result directory exists at: {RES_DIR}.")

# Define and create RES_DIR for the log file
LOG_DIR = os.path.join(WORKING_DIR, "tvd_mi")
os.makedirs(LOG_DIR, exist_ok=True)
print(f"Ensured logging directory exists at: {LOG_DIR}.")

Mounted at /content/drive
Ensured working directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/.
Ensured raw directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/iclr/final.
Ensured llm review directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews.
Ensured result directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/tvd_mi.
Ensured logging directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/tvd_mi.


In [5]:
# @title Setup definitions

### --- Settings --- ###

SIMULATION_ACTIVE = False # SIMULATION_ACTIVE = False activates the API calls

# Define the type of TVD-MI score to calculate using a numerical option
# 1: "paper / human review"
# 2: "paper / llm review"
# 3: "human review / llm review"
# 4: "llm review / llm review"
TVD_MI_SCORE_OPTION = 2

SEED_VALUE = 7 # Random

# Subsetting for testing
SUBSETTING_ACTIVE = False # Set to True to activate subsetting
NUM_SUBSET = 2 # Number of papers per group

### --- PARAMETERS --- ###

# Model
MODEL = "gpt-4o-mini-2024-07-18" # Robertson and Kayejo only specified "gpt-4o-mini"

### --- Content --- ####

# Target string to remove metadata from the papers full text
HEADER_TO_REMOVE = "GROBID - A machine learning software for extracting information from scholarly documents"

# Custom strings that should trigger a skip for the TVD-MI critique
SPECIAL_SKIP_STRINGS = {
    "ERROR: REVIEW GENERATION NOT SUCCESSFUL",
    "ERROR: UNKNOWN STATE REACHED",
}

# Message if TVD-MI critique yielded an error
TOTAL_FAILURE_MESSAGE = "LLM call failed after multiple retries"

# Message if TVD-MI critique was not attempted in the first place
SKIP_MESSAGE = "Skipped cause of invalid input"

# Define parameter columns for grouping and logging
PARAM_COLS = ["model", "temperature", "reasoning_effort", "chain_of_thought", "note_taking"]

# Define result file columns
RES_COL = [
    'paper_id',
] + PARAM_COLS + [
    'tpr_tvd_mi_score_1', 'tpr_tvd_mi_score_2', 'tpr_avg_tvd_mi_score',
    'fpr_tvd_mi_score_1', 'fpr_tvd_mi_score_2', 'fpr_avg_tvd_mi_score',
    'tvd_mi_score_option',
    'fpr_response_a_source_paper_id',
    'fpr_response_a_source_column',
    'tpr_raw_llm_response_1', 'tpr_raw_llm_response_2',
    'fpr_raw_llm_response_1', 'fpr_raw_llm_response_2',
    'is_created'
]

# Define log file columns
LOG_COL = ['combo', 'TPR_calculated', 'FPR_calculated', 'complete']

### --- API/Client --- ###

TEMPERATURE = 0.0 # As Robertson and Koyejo (2025) (in their repo)
MAX_TOKENS = 2000 # As Robertson and Koyejo (2025) (in their repo)
MAX_RETRIES = 3 # Random
RETRY_DELAY = 1.5 # Random

# API Keys
from google.colab import userdata
OPENAI_KEY = userdata.get('OPENAI_API_KEY')


In [6]:
# @title Data import

# Read dataset_paper
dataset_paper = pd.read_parquet(os.path.join(PAPER_DIR, "dataset_paper.parquet"))

# Select 'paper_id' and the target columns for paper content and human reviews
paper_content = dataset_paper[['paper_id', 'parsed_text', 'human_review_1', 'human_review_2', 'human_review_3']].copy()

# Remove metadata from the papers full text (ensure no shortcut identification for the TVD-MI critique exists)
paper_content['parsed_text'] = paper_content['parsed_text'].str.replace(HEADER_TO_REMOVE, '', regex=False).str.strip()

# Check paper_content
display(paper_content.head(3))
display(paper_content.shape)

# Read llm reviews
dataset_llm = pd.read_parquet(os.path.join(LLM_DIR, "llm_reviews_results.parquet"))

# Check dataset_llm
display(dataset_llm.head(3))
display(dataset_llm.shape)

,paper_id,parsed_text,human_review_1,human_review_2,human_review_3
0,vuD2xEtxZcj,"In deep learning, fine-grained N:M sparsity r...",Summary Of The Paper:\n\nThe paper aims to acc...,Summary Of The Paper:\n\nThis paper develops a...,Summary Of The Paper:\n\nThe paper presents a ...
1,WoByU5W5te0,We present a novel framework to regularize Ne...,Summary Of The Paper:\n\nThis paper proposes a...,Summary Of The Paper:\n\nThis paper focus on t...,Summary Of The Paper:\n\nThe paper proposes a ...
2,XZRmNjUMj0c,Neural Architecture Search (NAS) is a fast-de...,Summary Of The Paper:\n\nThis paper proposes a...,Summary Of The Paper:\n\nThis paper focused on...,Summary Of The Paper:\n\nThe paper introduces ...


(98, 5)

,paper_id,model,temperature,reasoning_effort,chain_of_thought,note_taking,llm_notes_1,llm_review_1
0,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Faithful,"Here is a concise, faithful summary of the pap...",# Summary Of The Paper\nThis paper studies app...
1,WoByU5W5te0,gpt-5-mini-2025-08-07,NaN,low,,Faithful,"Below is a faithful, concise summary of the Ge...",# Summary Of The Paper\nThe paper (GeCoNeRF / ...
2,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Objective,"Here is a concise, self-contained summary of t...",# Summary Of The Paper\nThis paper studies N:M...


(312, 8)

In [7]:
# @title Merge data

merged_df = pd.merge(paper_content, dataset_llm, on='paper_id', how='inner')
display(merged_df.head(3))
display(merged_df.shape)

,paper_id,parsed_text,human_review_1,human_review_2,human_review_3,model,temperature,reasoning_effort,chain_of_thought,note_taking,llm_notes_1,llm_review_1
0,vuD2xEtxZcj,"In deep learning, fine-grained N:M sparsity r...",Summary Of The Paper:\n\nThe paper aims to acc...,Summary Of The Paper:\n\nThis paper develops a...,Summary Of The Paper:\n\nThe paper presents a ...,gpt-5-mini-2025-08-07,NaN,low,,Faithful,"Here is a concise, faithful summary of the pap...",# Summary Of The Paper\nThis paper studies app...
1,vuD2xEtxZcj,"In deep learning, fine-grained N:M sparsity r...",Summary Of The Paper:\n\nThe paper aims to acc...,Summary Of The Paper:\n\nThis paper develops a...,Summary Of The Paper:\n\nThe paper presents a ...,gpt-5-mini-2025-08-07,NaN,low,,Objective,"Here is a concise, self-contained summary of t...",# Summary Of The Paper\nThis paper studies N:M...
2,vuD2xEtxZcj,"In deep learning, fine-grained N:M sparsity r...",Summary Of The Paper:\n\nThe paper aims to acc...,Summary Of The Paper:\n\nThis paper develops a...,Summary Of The Paper:\n\nThe paper presents a ...,gpt-5-mini-2025-08-07,NaN,low,,Evaluation,Thank you — below is a focused evaluation of t...,# Summary Of The Paper\nThe paper studies appl...


(312, 12)

In [8]:
try:
    if SUBSETTING_ACTIVE:
        print(f"Subsetting is ACTIVE: Applying to dataframes based on TVD_MI_SCORE_OPTION.", flush=True)

        if TVD_MI_SCORE_OPTION == 1:
            paper_content = paper_content.head(NUM_SUBSET).reset_index(drop=True)
        else: # TVD_MI_SCORE_OPTION is 2, 3, or 4
            group_cols = ['paper_id'] + PARAM_COLS

            # Ensure group_cols only contain columns present in merged_df
            actual_group_cols = [col for col in group_cols if col in merged_df.columns]

            if not actual_group_cols:
                print("Warning: No valid grouping columns found for subsetting merged_df. Subsetting will not be applied.", flush=True)
            else:
                merged_df = merged_df.groupby(actual_group_cols, dropna=False).head(NUM_SUBSET).reset_index(drop=True)
    else:
        print("Subsetting is INACTIVE: Using full dataframes.", flush=True)

except Exception as e:
    print(f"An error occurred during subsetting: {e}", flush=True)

# Display the head and shape of both dataframes to show the effect of subsetting
print("\n--- Current state of paper_content after subsetting ---")
display(paper_content.head(3))
display(paper_content.shape)

print("\n--- Current state of merged_df after subsetting ---")
display(merged_df.head(3))
display(merged_df.shape)

Subsetting is INACTIVE: Using full dataframes.

--- Current state of paper_content after subsetting ---


,paper_id,parsed_text,human_review_1,human_review_2,human_review_3
0,vuD2xEtxZcj,"In deep learning, fine-grained N:M sparsity r...",Summary Of The Paper:\n\nThe paper aims to acc...,Summary Of The Paper:\n\nThis paper develops a...,Summary Of The Paper:\n\nThe paper presents a ...
1,WoByU5W5te0,We present a novel framework to regularize Ne...,Summary Of The Paper:\n\nThis paper proposes a...,Summary Of The Paper:\n\nThis paper focus on t...,Summary Of The Paper:\n\nThe paper proposes a ...
2,XZRmNjUMj0c,Neural Architecture Search (NAS) is a fast-de...,Summary Of The Paper:\n\nThis paper proposes a...,Summary Of The Paper:\n\nThis paper focused on...,Summary Of The Paper:\n\nThe paper introduces ...


(98, 5)


--- Current state of merged_df after subsetting ---


,paper_id,parsed_text,human_review_1,human_review_2,human_review_3,model,temperature,reasoning_effort,chain_of_thought,note_taking,llm_notes_1,llm_review_1
0,vuD2xEtxZcj,"In deep learning, fine-grained N:M sparsity r...",Summary Of The Paper:\n\nThe paper aims to acc...,Summary Of The Paper:\n\nThis paper develops a...,Summary Of The Paper:\n\nThe paper presents a ...,gpt-5-mini-2025-08-07,NaN,low,,Faithful,"Here is a concise, faithful summary of the pap...",# Summary Of The Paper\nThis paper studies app...
1,vuD2xEtxZcj,"In deep learning, fine-grained N:M sparsity r...",Summary Of The Paper:\n\nThe paper aims to acc...,Summary Of The Paper:\n\nThis paper develops a...,Summary Of The Paper:\n\nThe paper presents a ...,gpt-5-mini-2025-08-07,NaN,low,,Objective,"Here is a concise, self-contained summary of t...",# Summary Of The Paper\nThis paper studies N:M...
2,vuD2xEtxZcj,"In deep learning, fine-grained N:M sparsity r...",Summary Of The Paper:\n\nThe paper aims to acc...,Summary Of The Paper:\n\nThis paper develops a...,Summary Of The Paper:\n\nThe paper presents a ...,gpt-5-mini-2025-08-07,NaN,low,,Evaluation,Thank you — below is a focused evaluation of t...,# Summary Of The Paper\nThe paper studies appl...


(312, 12)

### API

In [9]:
def generate_tvd_mi_prompt(response_a: str, response_b: str) -> str:
    """Generate prompt for TVD-MI critic"""
    prompt = f"""You are evaluating whether two responses could be from agents working on the same task.

Task description: Scientific paper peer review task

Response A: {response_a}

Response B: {response_b}

Do these responses show evidence of coming from the same task/source? Consider:
- Shared specific details, facts, or entities
- Similar topics or themes
- Overlapping information that would be unlikely if from different sources

Respond in two sentences or less. Put the information gain at the end in brackets in this format:

Rate the information gain:
- [[Significant Gain]]: Clear evidence they're from the same source (many shared specifics)
- [[Little Gain]]: Some shared elements but also differences
- [[No Gain]]: No evidence of shared source (could be from completely different tasks)"""

    return prompt

class LLMClient:
    def __init__(self, simulate: bool = True, seed: int = 7):
        """Initialize LLMClient with defaults"""
        self.simulate = simulate
        self.rng = random.Random(seed)
        self.cache = {}

        # Placeholder for real client when simulation=False:
        self._openai = None

    def _maybe_init_clients(self):
        """Start simulation/API clients"""
        # Simulation
        if self.simulate:
            return
        # Initialize OpenAI client
        if self._openai is None and "OPENAI_KEY" in globals():
            self._openai = OpenAI(api_key=OPENAI_KEY)

    def call(
        self,
        prompt: str,
        model: str = MODEL,
        temperature: float = TEMPERATURE,
        max_tokens: int = MAX_TOKENS,
        max_retries: int = MAX_RETRIES,
        retry_delay: float = RETRY_DELAY,
    ) -> str:
        """
        Defines the API calls / simulations according to the given prompt.
        Returns: The TVD-MI critique as a string.
        """
        self._maybe_init_clients()

        # In simulation
        if self.simulate:
            # Create artificial response options
            possible_responses = [
                "[significant gain]",
                "[little gain]",
                "[no gain]",
                "[total failure]"
            ]
            # Randomly choose one option
            simulated_response = self.rng.choice(possible_responses)
            return simulated_response

        # Call API
        for attempt in range(1, max_retries + 1):
            try:
                # Call client and return the critique
                resp = self._openai.chat.completions.create(
                    model=model,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=temperature,
                    max_tokens=max_tokens
                )
                # Extract the critique
                response_text = resp.choices[0].message.content
                return response_text

            except Exception as e:
                print(f"[LLM ERROR]: {e}", flush = True)
                if attempt == max_retries:
                    return TOTAL_FAILURE_MESSAGE
                time.sleep(retry_delay)
        return TOTAL_FAILURE_MESSAGE

def interpret_tvd_mi_response(response: str) -> str:
    """Convert LLM response to numeric score (as string) or return specific message strings."""
    # Standardize LLM response
    response = response.strip().lower()

    if "[significant gain]" in response:
        return "1.0"
    elif "[little gain]" in response:
        return "0.25"
    elif "[no gain]" in response:
        return "0.0"
    elif "[total failure]" in response:
        return TOTAL_FAILURE_MESSAGE
    else:
        # Default to no gain if response is unclear, and return as string
        print(f"Warning: Unclear response '{response}', defaulting to [[No Gain]]")
        return "0.0"

### Helpers for logging, status output and keys

In [10]:
def _nan_to_none(x):
  """Helper to handle NaN values when creating hashable keys for comparison"""
  try:
    if pd.isna(x):
      return None
  except Exception:
    pass
  return x

def get_unique_combo_key(row, param_cols):
  """Helper to create a unique hashable key for a paper-parameter combination"""
  key_elements = [row['paper_id']]
  for col in param_cols:
    key_elements.append(_nan_to_none(row[col]))
  return tuple(key_elements)

def format_param_strings(paper_id_val, param_dict):
  """Helper to format parameter string for logging and printing"""
  # For log file: paper_id=X|model=Y|temp=Z...
  log_parts = [f"paper_id={paper_id_val}"]
  for col in PARAM_COLS:
    value = param_dict.get(col)
    log_parts.append(f"{col}={_nan_to_none(value)}")
  combo_str = "|".join(log_parts)

  # For print message: model=Y, temperature=Z...
  print_param_parts = []
  for col in PARAM_COLS:
    value = param_dict.get(col)
    print_param_parts.append(f"{col}={'' if pd.isna(value) else value}")
  print_params_str = ", ".join(print_param_parts)

  return combo_str, print_params_str

def _parse_val_from_log_str(val_str):
  """Helper to parse values from the combo string to their original values"""
  if val_str == 'None':
    return None
  try:
    # Attempt float conversion for numbers (e.g., temperature)
    return float(val_str)
  except ValueError:
    return val_str # If not a float, return as string (e.g., 'low', 'high')

def get_unique_combo_key_from_log_entry(combo_str_from_log, param_cols):
  """Helper to create a unique hashable key from the combo string for log file parsing"""
  key_elements = []
  parts = combo_str_from_log.split('|')

  # Extract paper_id from the first part
  paper_id_part = parts[0]
  paper_id = paper_id_part.split('=', 1)[1]
  key_elements.append(paper_id)

  # Extract parameter values from the remaining parts
  param_map = {}
  for part in parts[1:]:
    key, val_str = part.split('=', 1)
    param_map[key] = _parse_val_from_log_str(val_str)

  # Assemble key_elements in the same order as get_unique_combo_key expects
  for col in param_cols:
    key_elements.append(param_map.get(col))

  return tuple(key_elements)

def is_numeric_string(s):
  """Helper to check if a string represents a numeric value"""
  if not isinstance(s, str):
    return False
  try:
    float(s)
    return True
  except ValueError:
    return False

### Configuration for TVD-MI score type settings

In [11]:
def configure_tvd_mi_run(option: int, res_dir: str, log_dir: str) -> Tuple[List[str], str, str]:
    """
    Configures the column pair and output paths based on the TVD-MI score option.
    Selects review columns deterministically when multiple are available.
    Returns: A tuple of (column_pair, output_path, log_path).
    """
    column_pair = []
    file_suffix = ""
    score_type_label = ""

    def _review_col_sort_key(col_name: str) -> int:
        suffix = col_name.rsplit('_', 1)[-1]
        return int(suffix) if suffix.isdigit() else -1

    # Dynamically get all human review columns
    all_human_review_cols = sorted(
        [col for col in paper_content.columns if col.startswith('human_review_')],
        key=_review_col_sort_key,
    )
    # Dynamically get all LLM review columns
    all_llm_review_cols = sorted(
        [col for col in dataset_llm.columns if col.startswith('llm_review_')],
        key=_review_col_sort_key,
    )

    # Select one human review column if available (first = canonical)
    selected_human_review_col = all_human_review_cols[0] if all_human_review_cols else ""
    # Select one LLM review column if available (last = latest generated review column)
    selected_llm_review_col = all_llm_review_cols[-1] if all_llm_review_cols else ""

    if option == 1:
        score_type_label = "paper / human review"
        if not selected_human_review_col:
            raise ValueError("No human review columns available for 'paper / human review' comparison.")
        column_pair = ["parsed_text", selected_human_review_col]
        file_suffix = f"paper_{selected_human_review_col}"

    elif option == 2:
        score_type_label = "paper / llm review"
        if not selected_llm_review_col:
            raise ValueError("No LLM review columns available for 'paper / llm review' comparison.")
        column_pair = ["parsed_text", selected_llm_review_col]
        file_suffix = f"paper_{selected_llm_review_col}"

    elif option == 3:
        score_type_label = "human review / llm review"
        if not selected_human_review_col or not selected_llm_review_col:
            raise ValueError("Not enough human/LLM review columns available for 'human review / llm review' comparison.")
        column_pair = [selected_human_review_col, selected_llm_review_col]
        file_suffix = f"h_{selected_human_review_col}_l_{selected_llm_review_col}"

    elif option == 4:
        score_type_label = "llm review / llm review"
        # Check if there are sufficient llm reviews for this case
        if len(all_llm_review_cols) < 2:
            raise ValueError("Not enough LLM review columns available for 'llm review / llm review' comparison (need at least two distinct ones).")

        # Select the latest and second-latest distinct LLM review columns
        selected_llm_review_col_1 = all_llm_review_cols[-1]
        selected_llm_review_col_2 = all_llm_review_cols[-2]

        column_pair = [selected_llm_review_col_1, selected_llm_review_col_2]
        file_suffix = f"llm_{selected_llm_review_col_1}_llm_{selected_llm_review_col_2}"
    else:
        raise ValueError(f"Invalid TVD_MI_OPTION: {option}. Please choose between 1 and 4.")

    # Create output and log paths
    output_path = os.path.join(res_dir, f"tvd_mi_scores_{file_suffix}.parquet")
    log_path = os.path.join(log_dir, f"tvd_mi_log_{file_suffix}.parquet")

    return column_pair, output_path, log_path

### Higher order Functions

In [12]:
def calculate_tvd_mi_scores_for_pair(llm_client, response_a_text: str, response_b_text: str) -> dict:
    """
    Generates the prompts, calls the client, and interprets scores for a given pair.
    All processes are balanced and the resulting scores are averaged.
    Returns: A dictionary containing the pair content, prompt content, balanced and total TVD-MI critiques.
    """
    # --- A, B ---

    # Generate the TVD-MI prompt
    tvd_mi_prompt_1 = generate_tvd_mi_prompt(response_a_text, response_b_text)

    # Call the client
    llm_critic_response_text_1 = llm_client.call(
        prompt=tvd_mi_prompt_1,
        model=MODEL,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS
    )

    # Interpret response
    tvd_mi_score_1_str = interpret_tvd_mi_response(llm_critic_response_text_1)

    # --- B, A ---

    # Generate the TVD-MI prompt
    tvd_mi_prompt_2 = generate_tvd_mi_prompt(response_b_text, response_a_text)

    # Call the client
    llm_critic_response_text_2 = llm_client.call(
        prompt=tvd_mi_prompt_2,
        model=MODEL,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS
    )

    # Interpret response
    tvd_mi_score_2_str = interpret_tvd_mi_response(llm_critic_response_text_2)

    # --- Results ---

    # Handle potential skip/failure messages
    if tvd_mi_score_1_str == SKIP_MESSAGE or tvd_mi_score_2_str == SKIP_MESSAGE:
        avg_tvd_mi_score_str = SKIP_MESSAGE
    elif tvd_mi_score_1_str == TOTAL_FAILURE_MESSAGE or tvd_mi_score_2_str == TOTAL_FAILURE_MESSAGE:
        avg_tvd_mi_score_str = TOTAL_FAILURE_MESSAGE
    else:
        # Calculate average score
        try:
            avg_tvd_mi_score = (float(tvd_mi_score_1_str) + float(tvd_mi_score_2_str)) / 2
            avg_tvd_mi_score_str = str(avg_tvd_mi_score)
        except ValueError:
            avg_tvd_mi_score_str = TOTAL_FAILURE_MESSAGE

    # Return responses, raw responses and scores
    return {
        'response_a': response_a_text,
        'response_b': response_b_text,
        'raw_response_1': llm_critic_response_text_1,
        'raw_response_2': llm_critic_response_text_2,
        'tvd_mi_score_1': tvd_mi_score_1_str,
        'tvd_mi_score_2': tvd_mi_score_2_str,
        'avg_tvd_mi_score': avg_tvd_mi_score_str
    }

In [13]:
def process_single_source(llm_client, row, response_a_col_name, response_b_col_name, full_source_df, param_cols):
    """
    Evaluates a single source, by calculating its TPR, FPR and success state.
    Returns: a tuple of (results_data_dict, log_data_dict).
    """
    # --- Initialize results for single row in result file ---

    # Add current paper id
    data_to_write = {'paper_id': row['paper_id']}

    # Add parameter values
    param_dict = {}
    for param_col in param_cols:
        param_dict[param_col] = row[param_col]
        data_to_write[param_col] = row[param_col]

    # Add TVD_MI_SCORE_OPTION to results
    data_to_write['tvd_mi_score_option'] = TVD_MI_SCORE_OPTION

    # Extract content from the current row for response_b (used for both TPR and FPR)
    response_b_text = str(row[response_b_col_name]) if pd.notna(row[response_b_col_name]) else ""

    # Check if response_b (llm review) was created successfully
    data_to_write['is_created'] = 0 if response_b_text in SPECIAL_SKIP_STRINGS else 1

    # --- TPR Calculation: response_a and response_b from the current row ---

    # Extract response_a for TPR from the current row
    response_a_tpr_text = str(row[response_a_col_name]) if pd.notna(row[response_a_col_name]) else ""

    # Check content
    if (not response_a_tpr_text or response_a_tpr_text in SPECIAL_SKIP_STRINGS) or (not response_b_text or response_b_text in SPECIAL_SKIP_STRINGS):
        # Assign skip messages
        data_to_write['tpr_tvd_mi_score_1'] = SKIP_MESSAGE
        data_to_write['tpr_tvd_mi_score_2'] = SKIP_MESSAGE
        data_to_write['tpr_avg_tvd_mi_score'] = SKIP_MESSAGE
        data_to_write['tpr_raw_llm_response_1'] = SKIP_MESSAGE
        data_to_write['tpr_raw_llm_response_2'] = SKIP_MESSAGE
    else:
        # Calculate scores
        tpr_results_raw = calculate_tvd_mi_scores_for_pair(llm_client, response_a_tpr_text, response_b_text)

        # Add TPR results
        data_to_write['tpr_tvd_mi_score_1'] = tpr_results_raw['tvd_mi_score_1']
        data_to_write['tpr_tvd_mi_score_2'] = tpr_results_raw['tvd_mi_score_2']
        data_to_write['tpr_avg_tvd_mi_score'] = tpr_results_raw['avg_tvd_mi_score']
        data_to_write['tpr_raw_llm_response_1'] = tpr_results_raw['raw_response_1']
        data_to_write['tpr_raw_llm_response_2'] = tpr_results_raw['raw_response_2']

    # --- FPR Calculation: response_b from current row, response_a from random different row ---

    # Filter to ensure that the randomly selected row has a different paper_id
    # and belongs to the same parameter configuration group.
    eligible_rows_for_a = full_source_df[full_source_df['paper_id'] != row['paper_id']]
    for param_col in param_cols:
        row_param_val = row[param_col]
        if pd.isna(row_param_val):
            eligible_rows_for_a = eligible_rows_for_a[eligible_rows_for_a[param_col].isna()]
        else:
            eligible_rows_for_a = eligible_rows_for_a[eligible_rows_for_a[param_col] == row_param_val]
    eligible_indices_for_a = eligible_rows_for_a.index.tolist()

    # Initialize source info for FPR (will be None if no eligible_indices or response_a_fpr_text is empty)
    fpr_response_a_source_paper_id = None
    fpr_response_a_source_column = None

    if not eligible_indices_for_a:
        # Take empty string
        response_a_fpr_text = ""
    else:
        # Choose a random different source
        random_a_idx = random.choice(eligible_indices_for_a)
        random_a_row = full_source_df.loc[random_a_idx]
        response_a_fpr_text = str(random_a_row[response_a_col_name]) if pd.notna(random_a_row[response_a_col_name]) else ""

        # Store the source metadata
        fpr_response_a_source_paper_id = random_a_row['paper_id']
        fpr_response_a_source_column = response_a_col_name

    # Add the source metadata to results
    data_to_write['fpr_response_a_source_paper_id'] = fpr_response_a_source_paper_id
    data_to_write['fpr_response_a_source_column'] = fpr_response_a_source_column

    # Check content
    if (not response_a_fpr_text or response_a_fpr_text in SPECIAL_SKIP_STRINGS) or (not response_b_text or response_b_text in SPECIAL_SKIP_STRINGS):
        # Assign skip messages
        data_to_write['fpr_tvd_mi_score_1'] = SKIP_MESSAGE
        data_to_write['fpr_tvd_mi_score_2'] = SKIP_MESSAGE
        data_to_write['fpr_avg_tvd_mi_score'] = SKIP_MESSAGE
        data_to_write['fpr_raw_llm_response_1'] = SKIP_MESSAGE
        data_to_write['fpr_raw_llm_response_2'] = SKIP_MESSAGE
    else:
        # Call scores
        fpr_results_raw = calculate_tvd_mi_scores_for_pair(llm_client, response_a_fpr_text, response_b_text)

        # Add FPR
        data_to_write['fpr_tvd_mi_score_1'] = fpr_results_raw['tvd_mi_score_1']
        data_to_write['fpr_tvd_mi_score_2'] = fpr_results_raw['tvd_mi_score_2']
        data_to_write['fpr_avg_tvd_mi_score'] = fpr_results_raw['avg_tvd_mi_score']
        data_to_write['fpr_raw_llm_response_1'] = fpr_results_raw['raw_response_1']
        data_to_write['fpr_raw_llm_response_2'] = fpr_results_raw['raw_response_2']

    # --- Determine completion status for logging ---

    # A calculation is considered 'calculated' for logging if its average score is a string representation of a number, SKIP_MESSAGE, or TOTAL_FAILURE_MESSAGE (libaral)
    tpr_calculated = (
        is_numeric_string(data_to_write['tpr_avg_tvd_mi_score']) or
        data_to_write['tpr_avg_tvd_mi_score'] == SKIP_MESSAGE or
        data_to_write['tpr_avg_tvd_mi_score'] == TOTAL_FAILURE_MESSAGE
    )
    fpr_calculated = (
        is_numeric_string(data_to_write['fpr_avg_tvd_mi_score']) or
        data_to_write['fpr_avg_tvd_mi_score'] == SKIP_MESSAGE or
        data_to_write['fpr_avg_tvd_mi_score'] == TOTAL_FAILURE_MESSAGE
    )

    # --- Logging ---

    # Log success states
    log_data_to_write = {
        'TPR_calculated': tpr_calculated,
        'FPR_calculated': fpr_calculated,
        'complete': tpr_calculated and fpr_calculated
    }

    # Format the combo string for logging
    combo_str, _ = format_param_strings(row['paper_id'], param_dict)
    log_data_to_write['combo'] = combo_str

    return data_to_write, log_data_to_write

### TVD-MI score calculation

In [14]:
if __name__ == "__main__":

    # Set global seed
    random.seed(SEED_VALUE)

    # Configure run
    column_pair, result_path, log_path = configure_tvd_mi_run(TVD_MI_SCORE_OPTION, RES_DIR, LOG_DIR)

    # Select prompt input columns based on column_pair
    response_a_col_name = column_pair[0]
    response_b_col_name = column_pair[1]

    # Print information
    print(f"Prompt input will use columns {response_a_col_name} and {response_b_col_name}", flush = True)
    print(f"Processing and saving results to: {result_path}", flush = True)
    print(f"Logging successful iterations to: {log_path}", flush = True)

    # Activate simulation/API calls
    llm_client = LLMClient(simulate=SIMULATION_ACTIVE, seed=SEED_VALUE)

    # Set temporary placeholders for the DataFrame and params to group by
    df_to_use_for_iteration = None
    current_param_cols_for_run = []

    if TVD_MI_SCORE_OPTION == 1:
        # For paper/human review, no grouping by parameters
        df_to_use_for_iteration = paper_content.copy().reset_index(drop=True)
        current_param_cols_for_run = []
        print(f"Option 1 ('paper / human review'): Iterating over 'paper_content' with no LLM parameters for grouping.", flush=True)
    else:
        # For other options, group by LLM parameters
        df_to_use_for_iteration = merged_df.copy().reset_index(drop=True)
        current_param_cols_for_run = PARAM_COLS
        print(f"Option {TVD_MI_SCORE_OPTION}: Iterating over 'merged_df' with parameters: {current_param_cols_for_run} for grouping.", flush=True)

    # Initialize keys
    done_keys = set()
    log_combos_count = 0
    result_combos_count = 0

    # Source 1: Load completed combos from the log file
    if os.path.exists(log_path):
        try:
            log_df = pd.read_parquet(log_path)
            # Filter for rows which are marked as complete
            completed_log_combos = log_df[(log_df['complete'] == True)]['combo'].tolist()
            log_combos_count = len(completed_log_combos)
            for combo_str in completed_log_combos:
                # Add combos to keys
                done_keys.add(get_unique_combo_key_from_log_entry(combo_str, current_param_cols_for_run))
            print(f"Loaded {log_combos_count} combos marked 'complete' from log file.", flush=True)
        except Exception as e:
            print(f"Error reading existing log parquet file: {e}. No combos loaded from log.", flush=True)
    else:
        print("No existing log file found. No combos loaded from log.", flush=True)

    # Source 2: Load combos with valid results from the result file
    if os.path.exists(result_path):
        try:
            result_df = pd.read_parquet(result_path)
            # Filter for rows where both average scores are valid (liberal)
            valid_result_rows = result_df[
                (result_df['tpr_avg_tvd_mi_score'].apply(lambda x: x == SKIP_MESSAGE or is_numeric_string(x) or x == TOTAL_FAILURE_MESSAGE)) &
                (result_df['fpr_avg_tvd_mi_score'].apply(lambda x: x == SKIP_MESSAGE or is_numeric_string(x) or x == TOTAL_FAILURE_MESSAGE))
            ]
            result_combos_count = len(valid_result_rows)

            for _, row in valid_result_rows.iterrows():
                # Add combos to keys
                done_keys.add(get_unique_combo_key(row, current_param_cols_for_run))
            print(f"Loaded {result_combos_count} combos with valid results from result file.", flush=True)
        except Exception as e:
            print(f"Error reading existing result parquet file: {e}. No combos loaded from results.", flush=True)
    else:
        print("No existing result file found. No combos loaded from results.", flush=True)

    print(f"Final count of already-completed combos (from log OR results): {len(done_keys)}", flush=True)

    # Adjust grouping logic based on whether there are parameters to group by
    grouped_iterable = []
    num_param_groups = 0

    if not current_param_cols_for_run:
        grouped_iterable = [(None, df_to_use_for_iteration)]
        num_param_groups = 1
        print(f"Processing as a single group (no explicit parameters).", flush=True)
    else:
        grouped_iterable = df_to_use_for_iteration.groupby(current_param_cols_for_run, dropna=False)
        num_param_groups = len(grouped_iterable)
        print(f"Starting processing for {num_param_groups} parameter configuration groups.", flush=True)

    # Initialize new_rows_count outside the loops for a global count
    new_rows_count = 0

    # Start iteration by group
    global_item_counter = 0

    # Outer loop: By group
    for group_idx, (params, group_df) in enumerate(tqdm(list(grouped_iterable), desc="Parameter Groups")):

        # Assigns current parameter values
        param_values_dict = dict(zip(current_param_cols_for_run, params)) if params is not None and current_param_cols_for_run else {}

        # Inner loop: By paper
        for i, row in tqdm(group_df.iterrows(), total=len(group_df), desc=f"  Papers in Group {group_idx+1}"):

            try:
                # Increment global counter
                global_item_counter += 1

                # Get paper ID
                paper_id = row['paper_id']

                # Get and print formatted parameter string
                _, print_params_str = format_param_strings(paper_id, param_values_dict)
                print(f"Processing Paper ID: {paper_id}, Parameters: {print_params_str}", flush=True)

                # Get current key
                current_combo_key = get_unique_combo_key(row, current_param_cols_for_run)

                # Check if current key is already present
                if current_combo_key in done_keys:
                    continue

                # Call the function to process each paper_id
                results_dict, log_dict = process_single_source(
                    llm_client, row, response_a_col_name,
                    response_b_col_name, group_df, current_param_cols_for_run
                )

                # Create DataFrame for current results
                df_result_entry = pd.DataFrame([results_dict], columns=RES_COL)

                # Create DataFrame for current log
                df_log_entry = pd.DataFrame([log_dict], columns=LOG_COL)

                if os.path.exists(result_path):
                    # Read, concat and save
                    existing_result_df = pd.read_parquet(result_path)
                    combined_result_df = pd.concat([existing_result_df, df_result_entry], ignore_index=True)
                    combined_result_df.to_parquet(result_path, index=False)
                else:
                    # Create a new result file
                    df_result_entry.to_parquet(result_path, index=False)

                # Update the global counter and add key immediately after successful result save (liberal)
                new_rows_count += len(df_result_entry)
                done_keys.add(current_combo_key)

                if os.path.exists(log_path):
                    try:
                        # Read, concat and save
                        existing_log_df = pd.read_parquet(log_path)
                        combined_log_df = pd.concat([existing_log_df, df_log_entry], ignore_index=True)
                        combined_log_df.to_parquet(log_path, index=False)

                    except Exception as e:
                        print(f"Warning: Could not append to existing log parquet file {log_path}: {e}.", flush=True)
                else:
                    # Write a new log file
                    df_log_entry.to_parquet(log_path, index=False)

            # Catch an inner loop error
            except Exception as e:
                error_combo_identifier = f"paper_id={paper_id}" if 'paper_id' in locals() else "unknown combo"
                print(f"[ERROR] {error_combo_identifier} -> {type(e).__name__}: {e}", flush=True)


Prompt input will use columns parsed_text and llm_review_1
Processing and saving results to: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/tvd_mi/tvd_mi_scores_paper_llm_review_1.parquet
Logging successful iterations to: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/tvd_mi/tvd_mi_log_paper_llm_review_1.parquet
Option 2: Iterating over 'merged_df' with parameters: ['model', 'temperature', 'reasoning_effort', 'chain_of_thought', 'note_taking'] for grouping.
Loaded 312 combos marked 'complete' from log file.
Loaded 312 combos with valid results from result file.
Final count of already-completed combos (from log OR results): 312
Starting processing for 156 parameter configuration groups.


Parameter Groups:   0%|          | 0/156 [00:00<?, ?it/s]

  Papers in Group 1:   0%|          | 0/2 [00:00<?, ?it/s]

Processing Paper ID: vuD2xEtxZcj, Parameters: model=gemini-3-flash-preview, temperature=, reasoning_effort=high, chain_of_thought=, note_taking=Evaluation
Processing Paper ID: WoByU5W5te0, Parameters: model=gemini-3-flash-preview, temperature=, reasoning_effort=high, chain_of_thought=, note_taking=Evaluation


  Papers in Group 2:   0%|          | 0/2 [00:00<?, ?it/s]

...

Processing Paper ID: vuD2xEtxZcj, Parameters: model=o4-mini-2025-04-16, temperature=, reasoning_effort=low, chain_of_thought=Explain your thought process step by step., note_taking=Evaluation
Processing Paper ID: WoByU5W5te0, Parameters: model=o4-mini-2025-04-16, temperature=, reasoning_effort=low, chain_of_thought=Explain your thought process step by step., note_taking=Evaluation


  Papers in Group 155:   0%|          | 0/2 [00:00<?, ?it/s]

Processing Paper ID: vuD2xEtxZcj, Parameters: model=o4-mini-2025-04-16, temperature=, reasoning_effort=low, chain_of_thought=Explain your thought process step by step., note_taking=Faithful
Processing Paper ID: WoByU5W5te0, Parameters: model=o4-mini-2025-04-16, temperature=, reasoning_effort=low, chain_of_thought=Explain your thought process step by step., note_taking=Faithful


  Papers in Group 156:   0%|          | 0/2 [00:00<?, ?it/s]

Processing Paper ID: vuD2xEtxZcj, Parameters: model=o4-mini-2025-04-16, temperature=, reasoning_effort=low, chain_of_thought=Explain your thought process step by step., note_taking=Objective
Processing Paper ID: WoByU5W5te0, Parameters: model=o4-mini-2025-04-16, temperature=, reasoning_effort=low, chain_of_thought=Explain your thought process step by step., note_taking=Objective


In [15]:
# result.parquet-file
if os.path.exists(result_path):
    try:
        tvd_mi_results_df = pd.read_parquet(result_path)
        # Check results only if loading was successful
        display(tvd_mi_results_df.head(5))
        display(tvd_mi_results_df.shape)
    except Exception as e:
        print(f"Warning: Could not read existing results parquet file {result_path}.")
else:
    print(f"Results file '{result_path}' does not exist.")

# log.parquet-file
if os.path.exists(log_path):
    try:
        tvd_mi_log_df = pd.read_parquet(log_path)
        # Check log only if loading was successful
        display(tvd_mi_log_df.head(5))
        display(tvd_mi_log_df.shape)
    except Exception as e:
        print(f"Warning: Could not read existing log parquet file {log_path}.")
else:
    print(f"Log file '{log_path}' does not exist.")

,paper_id,model,temperature,reasoning_effort,chain_of_thought,note_taking,tpr_tvd_mi_score_1,tpr_tvd_mi_score_2,tpr_avg_tvd_mi_score,fpr_tvd_mi_score_1,fpr_tvd_mi_score_2,fpr_avg_tvd_mi_score,tvd_mi_score_option,fpr_response_a_source_paper_id,fpr_response_a_source_column,tpr_raw_llm_response_1,tpr_raw_llm_response_2,fpr_raw_llm_response_1,fpr_raw_llm_response_2,is_created
0,vuD2xEtxZcj,gemini-3-flash-preview,NaN,high,,Evaluation,1.0,1.0,1.0,0.0,0.0,0.0,2,WoByU5W5te0,parsed_text,Both responses discuss the same paper on N:M s...,Both responses discuss the application of N:M ...,The two responses do not show evidence of comi...,The two responses do not show evidence of comi...,1
1,WoByU5W5te0,gemini-3-flash-preview,NaN,high,,Evaluation,1.0,1.0,1.0,0.0,0.0,0.0,2,vuD2xEtxZcj,parsed_text,Both responses discuss the GeCoNeRF framework ...,Both responses discuss the GeCoNeRF framework ...,The two responses do not show evidence of comi...,The two responses do not show evidence of comi...,1
2,vuD2xEtxZcj,gemini-3-flash-preview,NaN,high,,Faithful,1.0,1.0,1.0,0.0,0.0,0.0,2,WoByU5W5te0,parsed_text,Both responses discuss the application of fine...,Both responses discuss the application of N:M ...,The two responses do not show evidence of comi...,The two responses discuss different topics wit...,1
3,WoByU5W5te0,gemini-3-flash-preview,NaN,high,,Faithful,1.0,1.0,1.0,0.0,0.0,0.0,2,vuD2xEtxZcj,parsed_text,Both responses discuss the GeCoNeRF framework ...,"Both responses discuss the same framework, GeC...",The two responses do not show evidence of comi...,The two responses do not show evidence of comi...,1
4,vuD2xEtxZcj,gemini-3-flash-preview,NaN,high,,Objective,1.0,1.0,1.0,0.0,0.0,0.0,2,WoByU5W5te0,parsed_text,Both responses discuss the same paper on N:M s...,Both responses discuss the application of N:M ...,The two responses do not show evidence of comi...,The two responses do not show evidence of comi...,1


(312, 20)

,combo,TPR_calculated,FPR_calculated,complete
0,paper_id=vuD2xEtxZcj|model=gemini-3-flash-prev...,True,True,True
1,paper_id=WoByU5W5te0|model=gemini-3-flash-prev...,True,True,True
2,paper_id=vuD2xEtxZcj|model=gemini-3-flash-prev...,True,True,True
3,paper_id=WoByU5W5te0|model=gemini-3-flash-prev...,True,True,True
4,paper_id=vuD2xEtxZcj|model=gemini-3-flash-prev...,True,True,True


(312, 4)